# AI Programming — Lecture 11
## Lab 2-1: Ames Housing Regression

이번 실습에서는 **Ames Housing Dataset**을 이용하여 주택 가격을 예측하는 MLP 회귀 모델을 구성합니다.

단순히 하나의 모델을 학습하는 데서 끝나지 않고, 같은 데이터 분할과 전처리를 유지한 채
**regularization과 training technique을 하나씩 바꾸어** generalization 성능이 어떻게 변하는지 비교합니다.

### 학습 목표
- 수치형/범주형 feature가 함께 있는 tabular dataset을 전처리할 수 있습니다.
- missing value의 의미를 확인한 뒤 적절히 처리할 수 있습니다.
- one-hot encoding과 standardization에서 train set에만 `fit`해야 하는 이유를 설명할 수 있습니다.
- MLP regression에서 target scaling을 적용할 수 있습니다.
- Baseline, BatchNorm, Weight Decay, Dropout, LR scheduling을 비교할 수 있습니다.
- validation MAE를 이용해 model과 hyperparameter를 선택할 수 있습니다.

### Colab 실행 안내
이 노트북은 **Google Colab** 기준입니다. A100을 가정하지 않습니다.

- 기본 Colab GPU(T4 등) 또는 CPU에서도 실행할 수 있습니다.
- Ames dataset은 작기 때문에 `batch_size=32`를 그대로 사용합니다.
- 최대 epoch는 300으로 줄이고 early stopping을 사용하여 불필요한 학습을 줄였습니다.
- 데이터 파일: `MyDrive/Colab Notebooks/data/ames_train.csv`

> 실험에서는 한 번에 여러 요소를 바꾸기보다 **한 요소씩 변경**하여 성능 변화의 원인을 확인하세요.

## 1. 실습 환경 설정

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.feature_selection import SelectKBest, f_regression

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, AdamW

SEED = 42
BATCH_SIZE = 32
MAX_EPOCHS = 300
PATIENCE = 20


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.keras.backend.clear_session()


reset_seed()
print('TensorFlow version:', tf.__version__)

# Colab runtime 확인
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus[0].name if gpus else '사용하지 않음 (CPU)')

## 2. 데이터 불러오기와 기본 확인

`Id`는 단순 식별자이므로 입력 feature에서 제외합니다.  
`SalePrice`가 예측해야 할 target입니다.

먼저 데이터 크기, 자료형, missing value를 확인합니다.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running on Google Colab. Set DATA_PATH to your local file path.')

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/data/ames_train.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Cannot find {DATA_PATH}. Update DATA_PATH to the location of ames_train.csv.'
    )

df = pd.read_csv(DATA_PATH)

print('Data shape:', df.shape)
display(df.head())
print('\nData types:')
display(df.dtypes.value_counts().to_frame('count'))
print('\nTop missing-value columns:')
display(df.isna().sum().sort_values(ascending=False).head(20).to_frame('missing_count'))

## 3. Feature와 Target 분리

고정된 **60 / 20 / 20** 비율을 사용합니다.

- Train: preprocessing object fitting + model training
- Validation: hyperparameter 선택 + early stopping
- Test: 최종 generalization 성능 평가

`MSSubClass`는 숫자로 저장되어 있지만 주택 유형을 나타내는 **categorical code**이므로 문자열로 변환합니다.

In [ ]:
X = df.drop(columns=['Id', 'SalePrice']).copy()
y = df['SalePrice'].copy()

# MSSubClass is stored as a number, but it represents a housing type code.
X['MSSubClass'] = X['MSSubClass'].astype(str)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=SEED
)

split_info = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'samples': [len(X_train), len(X_val), len(X_test)],
    'ratio': [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)],
})

display(split_info.style.format({'ratio': '{:.1%}'}))

## 4. Missing Value 처리

모든 `NaN`이 단순한 "알 수 없음"을 뜻하지는 않습니다.

예를 들어 `PoolQC`, `GarageType`, `BsmtQual` 등의 결측은 해당 시설이 **존재하지 않음**을 의미할 수 있습니다.

이번 실습에서는 다음 규칙을 사용합니다.

- 시설 부재를 의미하는 categorical feature → `'None'`
- 시설 부재를 의미하는 numerical feature → `0`
- `LotFrontage` → train-set median
- `Electrical` → train-set mode

중요한 원칙은 **median과 mode를 train set에서만 계산**하는 것입니다.

In [ ]:
ABSENCE_CAT_COLS = [
    'PoolQC', 'Alley', 'Fence', 'FireplaceQu', 'MiscFeature',
    'MasVnrType', 'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2'
]

ABSENCE_NUM_COLS = ['GarageYrBlt', 'MasVnrArea']


def handle_missing_values(X_train, X_val, X_test):
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()

    for data in [X_train, X_val, X_test]:
        data[ABSENCE_CAT_COLS] = data[ABSENCE_CAT_COLS].fillna('None')
        data[ABSENCE_NUM_COLS] = data[ABSENCE_NUM_COLS].fillna(0)

    lot_frontage_median = X_train['LotFrontage'].median()
    electrical_mode = X_train['Electrical'].mode()[0]

    for data in [X_train, X_val, X_test]:
        data['LotFrontage'] = data['LotFrontage'].fillna(lot_frontage_median)
        data['Electrical'] = data['Electrical'].fillna(electrical_mode)

    return X_train, X_val, X_test


X_train_clean, X_val_clean, X_test_clean = handle_missing_values(
    X_train, X_val, X_test
)

missing_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'remaining_missing_values': [
        X_train_clean.isna().sum().sum(),
        X_val_clean.isna().sum().sum(),
        X_test_clean.isna().sum().sum(),
    ]
})

display(missing_summary)

## 5. Categorical Encoding과 Numerical Standardization

전처리 규칙:

```text
Train      → fit_transform()
Validation → transform()
Test       → transform()
```

Validation/Test 정보를 이용해 encoder나 scaler를 `fit`하면 data leakage가 발생할 수 있습니다.

In [ ]:
def preprocess_features(X_train, X_val, X_test):
    cat_cols = X_train.select_dtypes(include='object').columns
    num_cols = X_train.select_dtypes(include='number').columns

    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    scaler = StandardScaler()

    X_train_cat = encoder.fit_transform(X_train[cat_cols])
    X_val_cat = encoder.transform(X_val[cat_cols])
    X_test_cat = encoder.transform(X_test[cat_cols])

    X_train_num = scaler.fit_transform(X_train[num_cols])
    X_val_num = scaler.transform(X_val[num_cols])
    X_test_num = scaler.transform(X_test[num_cols])

    X_train_input = np.concatenate([X_train_num, X_train_cat], axis=1)
    X_val_input = np.concatenate([X_val_num, X_val_cat], axis=1)
    X_test_input = np.concatenate([X_test_num, X_test_cat], axis=1)

    return {
        'X_train': X_train_input,
        'X_val': X_val_input,
        'X_test': X_test_input,
        'cat_cols': cat_cols,
        'num_cols': num_cols,
        'encoder': encoder,
        'scaler': scaler,
    }


features = preprocess_features(X_train_clean, X_val_clean, X_test_clean)
X_train_input = features['X_train']
X_val_input = features['X_val']
X_test_input = features['X_test']

input_dim = X_train_input.shape[1]

print('Numerical features:', len(features['num_cols']))
print('Categorical columns:', len(features['cat_cols']))
print('Final input dimension:', input_dim)

## 6. Target Scaling

주택 가격은 값의 범위가 크기 때문에 MSE를 그대로 사용하면 loss와 gradient의 크기가 커질 수 있습니다.

따라서 `SalePrice`도 **train set 기준으로 standardization**한 뒤 학습하고,
최종 평가에서는 다시 달러 단위로 복원합니다.

In [ ]:
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).flatten()
y_val_scaled = y_scaler.transform(y_val.to_numpy().reshape(-1, 1)).flatten()
y_test_array = y_test.to_numpy()


def inverse_target(y_scaled):
    return y_scaler.inverse_transform(np.asarray(y_scaled).reshape(-1, 1)).flatten()


print('Target mean from training set:', f'${y_scaler.mean_[0]:,.0f}')
print('Target std from training set:', f'${y_scaler.scale_[0]:,.0f}')

## 7. 공통 Model / Training / Evaluation 함수

이후 실험은 데이터 분할과 전처리를 모두 동일하게 유지합니다.

각 실험에서는 model의 **한 구성 요소만 변경**하여 어떤 선택이 generalization에 영향을 주는지 확인합니다.

In [ ]:
def build_mlp(
    input_dim,
    hidden_units=(64, 32),
    dropout_rate=0.0,
    use_batch_norm=False,
    optimizer='adam',
    learning_rate=2e-4,
    weight_decay=0.0,
):
    layers = [Input(shape=(input_dim,))]

    for units in hidden_units:
        if use_batch_norm:
            layers.append(Dense(units, use_bias=False, kernel_initializer='he_normal'))
            layers.append(BatchNormalization())
            layers.append(Activation('relu'))
        else:
            layers.append(Dense(units, activation='relu', kernel_initializer='he_normal'))

        if dropout_rate > 0:
            layers.append(Dropout(dropout_rate))

    layers.append(Dense(1))

    model = Sequential(layers)

    if optimizer == 'adamw':
        opt = AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
    else:
        opt = Adam(learning_rate=learning_rate)

    model.compile(loss='mse', optimizer=opt, metrics=['mae'])
    return model


def make_callbacks(use_lr_scheduler=False, patience=PATIENCE):
    callbacks = []

    if use_lr_scheduler:
        callbacks.append(
            ReduceLROnPlateau(
                monitor='val_mae', factor=0.5, patience=15,
                cooldown=5, min_lr=1e-5, mode='min', verbose=1
            )
        )

    callbacks.append(
        EarlyStopping(
            monitor='val_mae', patience=patience, mode='min',
            restore_best_weights=True, verbose=1
        )
    )

    return callbacks


def evaluate_model(model, X_test_input, y_test_array):
    y_pred_scaled = model.predict(X_test_input, verbose=0).flatten()
    y_pred = inverse_target(y_pred_scaled)
    mse = mean_squared_error(y_test_array, y_pred)

    return {
        'test_mae': mean_absolute_error(y_test_array, y_pred),
        'test_rmse': np.sqrt(mse),
        'y_pred': y_pred,
    }


def run_experiment(
    name,
    X_train_input=X_train_input,
    X_val_input=X_val_input,
    X_test_input=X_test_input,
    y_train_scaled=y_train_scaled,
    y_val_scaled=y_val_scaled,
    y_test_array=y_test_array,
    seed=SEED,
    **model_kwargs,
):
    reset_seed(seed)

    use_lr_scheduler = model_kwargs.pop('use_lr_scheduler', False)
    model = build_mlp(input_dim=X_train_input.shape[1], **model_kwargs)

    history = model.fit(
        X_train_input, y_train_scaled,
        validation_data=(X_val_input, y_val_scaled),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(use_lr_scheduler),
        verbose=0,
    )

    best_epoch = int(np.argmin(history.history['val_mae']) + 1)
    best_val_mae = min(history.history['val_mae']) * y_scaler.scale_[0]
    evaluation = evaluate_model(model, X_test_input, y_test_array)

    record = {
        'model': name,
        'best_epoch': best_epoch,
        'best_val_mae': best_val_mae,
        'test_mae': evaluation['test_mae'],
        'test_rmse': evaluation['test_rmse'],
        'parameters': model.count_params(),
    }

    return model, history, evaluation['y_pred'], record


def plot_learning_and_prediction(history, y_pred, title):
    best_epoch = int(np.argmin(history.history['val_mae']) + 1)
    train_mae = np.array(history.history['mae']) * y_scaler.scale_[0]
    val_mae = np.array(history.history['val_mae']) * y_scaler.scale_[0]
    test_mae = mean_absolute_error(y_test_array, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(train_mae, label='Training MAE')
    axes[0].plot(val_mae, label='Validation MAE')
    axes[0].axvline(best_epoch - 1, linestyle='--', label='Best Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MAE ($)')
    axes[0].set_title('Learning Curve')
    axes[0].legend()
    axes[0].grid(True)

    min_price = min(y_test_array.min(), y_pred.min())
    max_price = max(y_test_array.max(), y_pred.max())
    axes[1].scatter(y_test_array, y_pred, alpha=0.6)
    axes[1].plot([min_price, max_price], [min_price, max_price], '--')
    axes[1].set_xlabel('Actual Sale Price')
    axes[1].set_ylabel('Predicted Sale Price')
    axes[1].set_title(f'{title}\nTest MAE: ${test_mae:,.0f}')
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

## 8. Experiment 1 — Baseline MLP

기준 모델입니다.

```text
Input
→ Dense(64, ReLU)
→ Dense(32, ReLU)
→ Dense(1)
```

이후 실험 결과는 모두 이 baseline과 비교합니다.

In [ ]:
results = []
experiments = {}

model_base, history_base, pred_base, record_base = run_experiment(
    'Baseline MLP',
    hidden_units=(64, 32),
)

results.append(record_base)
experiments['Baseline MLP'] = (model_base, history_base, pred_base)

display(pd.DataFrame(results))
plot_learning_and_prediction(history_base, pred_base, 'Baseline MLP')

## 9. Experiment 2 — Batch Normalization

각 hidden layer에서 다음 순서를 사용합니다.

```text
Dense → BatchNorm → ReLU
```

BatchNorm은 많은 문제에서 유용하지만,
Ames처럼 sample 수가 작고 one-hot feature가 많은 tabular dataset에서는 batch statistics가 불안정할 수 있습니다.

In [ ]:
model_bn, history_bn, pred_bn, record_bn = run_experiment(
    'BatchNorm',
    hidden_units=(64, 32),
    use_batch_norm=True,
)

results.append(record_bn)
experiments['BatchNorm'] = (model_bn, history_bn, pred_bn)

display(pd.DataFrame(results).sort_values('test_mae'))
plot_learning_and_prediction(history_bn, pred_bn, 'BatchNorm')

## 10. Experiment 3 — Weight Decay with AdamW

MLP 구조는 baseline과 동일하게 유지하고 optimizer만 AdamW로 변경합니다.

Weight decay는 지나치게 큰 weight를 억제하여 overfitting을 완화할 수 있습니다.

In [ ]:
model_wd, history_wd, pred_wd, record_wd = run_experiment(
    'AdamW weight decay',
    hidden_units=(64, 32),
    optimizer='adamw',
    weight_decay=1e-1,
)

results.append(record_wd)
experiments['AdamW weight decay'] = (model_wd, history_wd, pred_wd)

display(pd.DataFrame(results).sort_values('test_mae'))
plot_learning_and_prediction(history_wd, pred_wd, 'AdamW Weight Decay')

## 11. Experiment 4 — Dropout Rate 비교

Hidden layer 뒤에 dropout을 적용합니다.

이번 실습에서는 다음 세 값을 비교합니다.

```text
0.10
0.20
0.30
```

**Validation MAE가 가장 낮은 dropout rate**를 선택합니다.

In [ ]:
dropout_rates = [0.10, 0.20, 0.30]
dropout_records = []
dropout_experiments = {}

for rate in dropout_rates:
    model, history, pred, record = run_experiment(
        f'Dropout {rate:.2f}',
        hidden_units=(64, 32),
        dropout_rate=rate,
    )

    dropout_records.append(record)
    dropout_experiments[rate] = (model, history, pred, record)


dropout_df = pd.DataFrame(dropout_records).sort_values('best_val_mae')
display(dropout_df)

best_dropout_rate = float(dropout_df.iloc[0]['model'].split()[-1])
best_model, best_history, best_pred, best_record = dropout_experiments[best_dropout_rate]

results.append(best_record)
experiments[f'Dropout {best_dropout_rate:.2f}'] = (best_model, best_history, best_pred)

print('Selected dropout rate:', best_dropout_rate)
display(pd.DataFrame(results).sort_values('test_mae'))
plot_learning_and_prediction(best_history, best_pred, f'Dropout {best_dropout_rate:.2f}')

## 12. Experiment 5 — Dropout + Learning-Rate Scheduling

앞에서 선택한 best dropout model을 유지하면서,
validation MAE가 개선되지 않을 때 learning rate를 줄입니다.

단, scheduler가 항상 성능을 높이는 것은 아닙니다.
Early stopping과 best epoch의 관계를 함께 확인하세요.

In [ ]:
model_lr, history_lr, pred_lr, record_lr = run_experiment(
    f'Dropout {best_dropout_rate:.2f} + LR schedule',
    hidden_units=(64, 32),
    dropout_rate=best_dropout_rate,
    use_lr_scheduler=True,
)

results.append(record_lr)
experiments[f'Dropout {best_dropout_rate:.2f} + LR schedule'] = (model_lr, history_lr, pred_lr)

display(pd.DataFrame(results).sort_values('test_mae'))
plot_learning_and_prediction(history_lr, pred_lr, f'Dropout {best_dropout_rate:.2f} + LR schedule')

## 13. 선택 실습 — Log-Transformed Target

주택 가격은 오른쪽 꼬리가 긴 분포를 가집니다.

`log1p(SalePrice)`를 target으로 사용하면 고가 주택의 상대적 영향이 줄어듭니다.
하지만 **달러 단위 MAE가 반드시 개선되는 것은 아닙니다.**

In [ ]:
def run_log_target_experiment(dropout_rate=0.10, seed=SEED):
    reset_seed(seed)

    y_train_log = np.log1p(y_train.to_numpy())
    y_val_log = np.log1p(y_val.to_numpy())

    model = build_mlp(
        input_dim=input_dim,
        hidden_units=(64, 32),
        dropout_rate=dropout_rate,
    )

    history = model.fit(
        X_train_input, y_train_log,
        validation_data=(X_val_input, y_val_log),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(False),
        verbose=0,
    )

    y_pred_log = model.predict(X_test_input, verbose=0).flatten()
    y_pred = np.expm1(y_pred_log)

    record = {
        'model': 'Log target + Dropout',
        'best_epoch': int(np.argmin(history.history['val_mae']) + 1),
        'best_val_mae': np.nan,
        'test_mae': mean_absolute_error(y_test_array, y_pred),
        'test_rmse': np.sqrt(mean_squared_error(y_test_array, y_pred)),
        'parameters': model.count_params(),
    }

    return model, history, y_pred, record


model_log, history_log, pred_log, record_log = run_log_target_experiment(
    dropout_rate=best_dropout_rate
)

results.append(record_log)
experiments['Log target + Dropout'] = (model_log, history_log, pred_log)

display(pd.DataFrame(results).sort_values('test_mae'))

# The learning curve is in log-price units, so we inspect the prediction scatter instead.
plt.figure(figsize=(5, 5))
min_price = min(y_test_array.min(), pred_log.min())
max_price = max(y_test_array.max(), pred_log.max())
plt.scatter(y_test_array, pred_log, alpha=0.6)
plt.plot([min_price, max_price], [min_price, max_price], '--')
plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.title(f'Log Target + Dropout\nTest MAE: ${record_log["test_mae"]:,.0f}')
plt.grid(True)
plt.show()

## 14. 선택 실습 — Top-K Feature Selection

각 feature와 target의 개별적인 선형 관계를 기준으로 상위 feature만 선택합니다.

간단한 방법이지만 feature 간 **interaction**을 놓칠 수 있다는 한계가 있습니다.

Selector는 반드시 train set에서만 `fit`합니다.

In [ ]:
k_values = [20, 50, 100]
topk_records = []

for k in k_values:
    selector = SelectKBest(score_func=f_regression, k=k)

    X_train_k = selector.fit_transform(X_train_input, y_train_scaled)
    X_val_k = selector.transform(X_val_input)
    X_test_k = selector.transform(X_test_input)

    model, history, pred, record = run_experiment(
        f'Top-{k} features + Dropout',
        X_train_input=X_train_k,
        X_val_input=X_val_k,
        X_test_input=X_test_k,
        hidden_units=(64, 32),
        dropout_rate=best_dropout_rate,
    )

    topk_records.append(record)


topk_df = pd.DataFrame(topk_records).sort_values('best_val_mae')
display(topk_df)

best_topk_record = topk_df.iloc[0].to_dict()
results.append(best_topk_record)

display(pd.DataFrame(results).sort_values('test_mae'))

## 15. 선택 실습 — Smaller MLP

Hidden layer를

```text
64 → 32
```

에서

```text
32 → 16
```

으로 줄여 model capacity를 낮춥니다.

복잡도를 낮추면 overfitting이 줄 수 있지만, 동시에 표현 능력도 감소할 수 있습니다.

In [ ]:
model_small, history_small, pred_small, record_small = run_experiment(
    'Small MLP + Dropout',
    hidden_units=(32, 16),
    dropout_rate=best_dropout_rate,
)

results.append(record_small)
experiments['Small MLP + Dropout'] = (model_small, history_small, pred_small)

display(pd.DataFrame(results).sort_values('test_mae'))
plot_learning_and_prediction(history_small, pred_small, 'Small MLP + Dropout')

## 16. 최종 비교

실험 중 model 선택은 validation 성능을 기준으로 수행합니다.  
Test set은 최종 비교 단계에서만 사용해야 합니다.

In [ ]:
results_df = pd.DataFrame(results).drop_duplicates(subset=['model'], keep='last')
results_df = results_df.sort_values('test_mae').reset_index(drop=True)

display(
    results_df.style.format({
        'best_val_mae': '${:,.0f}',
        'test_mae': '${:,.0f}',
        'test_rmse': '${:,.0f}',
        'parameters': '{:,}',
    }).highlight_min(subset=['test_mae'], axis=0)
)

best_name = results_df.iloc[0]['model']
print('Best test MAE model:', best_name)

## 17. 결과 해석 체크리스트

다음 질문에 답해 보세요.

1. Baseline에서 training MAE와 validation MAE의 gap은 어떻게 변합니까?
2. Batch Normalization은 이 작은 tabular dataset에서 도움이 되었습니까?
3. Weight Decay만 적용했을 때 성능은 어떻게 변했습니까?
4. 어떤 dropout rate가 가장 좋았습니까?
5. LR scheduling은 best dropout model을 더 개선했습니까?
6. Log transform은 고가 주택 예측과 전체 MAE에 어떤 영향을 주었습니까?
7. Top-K feature selection은 왜 full feature보다 나쁠 수 있습니까?
8. Smaller MLP는 overfitting을 줄였습니까, 아니면 capacity 부족을 만들었습니까?

> 목표는 모든 technique을 한꺼번에 사용하는 것이 아니라, **어떤 technique이 왜 도움이 되는지 실험으로 확인하는 것**입니다.